# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GazalaNK/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Score each page by how far its actual CTR falls below the expected CTR for its position tier (the residual from w03/w04). Pages with impressions below a minimum threshold are excluded, since low-volume pages produce noisy, unreliable CTR numbers.

Score: -1 × ctr_residual (more negative residual → higher score → higher priority)

Reason code (one): ctr_below_tier_expectation — this page's CTR sits meaningfully below the median CTR of pages at the same position tier, despite having enough impressions to be a reliable signal.

Action label: review_metadata_and_snippet — the standard action for this lane per the guide: rewrite title/meta or review intent match.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# just a quick check that your feature frame (from w03/w04) is loaded
import duckdb
import pandas as pd
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"""
    CREATE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{hf_token}'
    );
""")

df_fact = con.sql("""
    SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet')
    WHERE gsc_data_available IS TRUE
""").df()

page = df_fact.groupby("content_hash_id").agg(
    impressions=("gsc_impressions","sum"),
    clicks=("gsc_clicks","sum"),
    avg_position=("gsc_avg_position","mean"),
).reset_index()
page["ctr"] = page["clicks"] / page["impressions"]
page["position_tier"] = pd.cut(page["avg_position"], bins=[0,3,10,20,1000], labels=["1-3","4-10","11-20","20+"])
page["tier_median_ctr"] = page.groupby("position_tier")["ctr"].transform("median")
page["ctr_residual"] = page["ctr"] - page["tier_median_ctr"]

print(page.shape)
page.head()

print(page.shape)
page.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(176738, 8)
(176738, 8)


/tmp/ipykernel_982/1508178551.py:32: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  page["tier_median_ctr"] = page.groupby("position_tier")["ctr"].transform("median")


,content_hash_id,impressions,clicks,avg_position,ctr,position_tier,tier_median_ctr,ctr_residual
0,content_000005d4ced12088,86,0,72.854861,0.000000,20+,0.0,0.000000
1,content_00007bd2985b77c3,47,0,5.269565,0.000000,4-10,0.0,0.000000
2,content_0000cd28fbda69f3,29,0,4.251282,0.000000,4-10,0.0,0.000000
3,content_0000d495bfbfb4a8,15,0,3.333333,0.000000,4-10,0.0,0.000000
4,content_00014efc121d911d,116,1,4.964683,0.008621,4-10,0.0,0.008621


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

MIN_IMPRESSIONS = 100  # minimum volume threshold to avoid noise

scored = page[page["impressions"] >= MIN_IMPRESSIONS].copy()
scored["score"] = -1 * scored["ctr_residual"]
scored["reason_code"] = "ctr_below_tier_expectation"
scored["action"] = "review_metadata_and_snippet"

ranked = scored.sort_values("score", ascending=False).reset_index(drop=True)

os.makedirs("work/outputs", exist_ok=True)
ranked.to_csv("work/outputs/baseline_action_score.csv", index=False)

print(ranked.shape)
ranked.head(20)


(101441, 11)


,content_hash_id,impressions,clicks,avg_position,ctr,position_tier,tier_median_ctr,ctr_residual,score,reason_code,action
0,content_fffff09da8a25da6,563,0,1.408799,0.0,1-3,0.0,0.0,-0.0,ctr_below_tier_expectation,review_metadata_and_snippet
1,content_fff61fa922a978fb,480,0,38.889875,0.0,20+,0.0,0.0,-0.0,ctr_below_tier_expectation,review_metadata_and_snippet
2,content_fff57dbceac30faa,304,0,44.891084,0.0,20+,0.0,0.0,-0.0,ctr_below_tier_expectation,review_metadata_and_snippet
3,content_fff2a166866a2605,214,0,4.012820,0.0,4-10,0.0,0.0,-0.0,ctr_below_tier_expectation,review_metadata_and_snippet
4,content_fff0583f20483b57,381,0,8.752991,0.0,4-10,0.0,0.0,-0.0,ctr_below_tier_expectation,review_metadata_and_snippet
5,content_ffed7a7bfa49197b,1428,0,10.681029,0.0,11-20,0.0,0.0,-0.0,ctr_below_tier_expectation,review_metadata_and_snippet
6,content_ffecf3edb8d0aa8a,368,0,19.307723,0.0,11-20,0.0,0.0,-0.0,ctr_below_tier_expectation,review_metadata_and_snippet
7,content_00157b715bf9db77,3335,0,5.269858,0.0,4-10,0.0,0.0,-0.0,ctr_below_tier_expectation,review_metadata_and_snippet
8,content_00121aaf9fd41914,315,0,12.970868,0.0,11-20,0.0,0.0,-0.0,ctr_below_tier_expectation,review_metadata_and_snippet
9,content_000f4b73532b9e9d,100,0,14.949691,0.0,11-20,0.0,0.0,-0.0,ctr_below_tier_expectation,review_metadata_and_snippet


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

top20 = ranked.head(20)
top20[["content_hash_id", "impressions", "clicks", "ctr", "avg_position", "ctr_residual", "score", "action"]]


,content_hash_id,impressions,clicks,ctr,avg_position,ctr_residual,score,action
0,content_fffff09da8a25da6,563,0,0.0,1.408799,0.0,-0.0,review_metadata_and_snippet
1,content_fff61fa922a978fb,480,0,0.0,38.889875,0.0,-0.0,review_metadata_and_snippet
2,content_fff57dbceac30faa,304,0,0.0,44.891084,0.0,-0.0,review_metadata_and_snippet
3,content_fff2a166866a2605,214,0,0.0,4.012820,0.0,-0.0,review_metadata_and_snippet
4,content_fff0583f20483b57,381,0,0.0,8.752991,0.0,-0.0,review_metadata_and_snippet
5,content_ffed7a7bfa49197b,1428,0,0.0,10.681029,0.0,-0.0,review_metadata_and_snippet
6,content_ffecf3edb8d0aa8a,368,0,0.0,19.307723,0.0,-0.0,review_metadata_and_snippet
7,content_00157b715bf9db77,3335,0,0.0,5.269858,0.0,-0.0,review_metadata_and_snippet
8,content_00121aaf9fd41914,315,0,0.0,12.970868,0.0,-0.0,review_metadata_and_snippet
9,content_000f4b73532b9e9d,100,0,0.0,14.949691,0.0,-0.0,review_metadata_and_snippet


flagged for CTR well below its position-tier median despite 800 impressions. Confidence: medium-high, plenty of volume. Would be wrong if this page recently had a title/meta change not yet reflected in ranking, or if it's seasonal content past its peak.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks: [Note any top-20 rows with borderline impression counts or unusual position outliers — inspect after running.]
Leakage check: No FlyRank product flags (health_score, priority_score, action_type) were used as inputs. No future-window data was used — all features come from the same month's observed impressions/clicks/position, known before any review decision.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.